# Bronze Layer: Raw Data Ingestion

## Objective
The **Bronze Layer** is the first stage of our medallion architecture. It serves as the raw landing zone where data from external sources is ingested with minimal transformation. Our goal here is to:

1. **Load the raw CSV data** from the Kaggle Superstore dataset
2. **Create the bronze schema** in DuckDB to organize our data
3. **Verify data ingestion** by checking row counts and previewing the data

**Why Bronze?** At this stage, we preserve all data as-is, including any quality issues. This gives us a complete audit trail and allows us to diagnose problems in downstream layers.

---

## Step 1: Initialize DuckDB Connection & Create Schema

First, we establish a connection to our DuckDB database and create a dedicated schema for bronze-layer tables. Using schemas helps organize our data warehouse into logical layers.

In [ ]:
import psycopg2
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# PostgreSQL connection parameters
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "retailion")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

# Create SQLAlchemy engine
engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

# Create a connection for executing SQL
con = engine.connect()

# Create the bronze schema for raw data
con.execute(text("CREATE SCHEMA IF NOT EXISTS bronze;"))
con.commit()

print("✅ Connected to PostgreSQL and created bronze schema")

## Step 2: Load Raw CSV Data

Now we ingest the raw superstore CSV file directly into the bronze schema. We use DuckDB's `read_csv_auto()` function which:
- **Automatically detects data types** (dates, numbers, strings)
- **Sets `ignore_errors=true`** to skip any malformed rows gracefully
- **Preserves column names** exactly as they appear in the CSV

This approach ensures we capture all usable data while being resilient to minor data quality issues.

In [ ]:
import pandas as pd

# Load CSV data using pandas
df = pd.read_csv('../data/Sample - Superstore.csv')

# Write to PostgreSQL bronze schema
df.to_sql('superstore', engine, schema='bronze', if_exists='replace', index=False)

print("✅ CSV data loaded into bronze.superstore")

## Step 3: Verify Data Ingestion

Let's validate that the data was loaded successfully by checking:
- **Total number of rows** ingested
- **Sample records** to visually inspect the data
- **Column structure** to understand what fields we're working with

In [ ]:
# Get row count
count = pd.read_sql("SELECT COUNT(*) as count FROM bronze.superstore", engine)['count'][0]
print(f"✅ Bronze Layer successfully processed! Total raw data: {count:,} rows.\n")

# Display first 3 rows
display(pd.read_sql("SELECT * FROM bronze.superstore LIMIT 3", engine))

## Results Summary

✅ **Data Ingestion Complete!**

- **Total Records:** 9,627 rows of superstore transaction data
- **Total Columns:** 21 fields covering customers, products, locations, and financial metrics
- **Data Types:** Mix of text, dates, and numeric values (exact types determined by DuckDB's auto-detection)

## What's Next?

The bronze layer now contains our raw data. In the **Silver Layer** (notebook 02_silver.ipynb), we will:
1. Explore and audit data quality (nulls, duplicates, distributions)
2. Apply type casting and standardization
3. Clean the data for analytical use
4. Perform exploratory data analysis (EDA) with visualizations

In [ ]:
# Close connection
con.close()
engine.dispose()